# Exploring EuroSAT (step E1b)

A look at the data before it is used. This notebook is **not** part of the pipeline: nothing
depends on it. The numbers that go into the report come from `mo433.data_stats`
(`make stats` writes `$MO433_DATA/eurosat/stats.json`); this notebook calls the same functions
and adds the figures.

Requires step E1 (`make dataset`). Splits below are the raw TorchGeo lists; our training split
is train + val.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from mo433 import config, data_stats
from mo433.dataset import load_eurosat

data = load_eurosat()
images, labels, classes, lists = data["images"], data["labels"], data["classes"], data["lists"]
stats = data_stats.compute_stats(data)
rng = np.random.default_rng(0)
print(images.shape, images.dtype, "| classes:", len(classes))

## 1. Samples per class
Eight random images per class (seed 0). Which classes look alike?

In [ ]:
n = 8
fig, axes = plt.subplots(len(classes), n, figsize=(n * 1.3, len(classes) * 1.4))
for k, name in enumerate(classes):
    for j, i in enumerate(rng.choice(np.flatnonzero(labels == k), n, replace=False)):
        ax = axes[k, j]
        ax.imshow(images[i])
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=9)
plt.suptitle("EuroSAT: 8 random images per class", fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Class counts per split
The TorchGeo split is not exactly stratified. `max_proportion_gap_pp` is the largest difference,
in percentage points, between a group's class proportions and those of the full dataset.

In [ ]:
counts = pd.DataFrame(stats["class_counts"])
counts.loc["total"] = counts.sum()
display(counts)
display(pd.Series(stats["max_proportion_gap_pp"], name="max proportion gap (pp)").to_frame())

In [ ]:
props = pd.DataFrame(stats["class_counts"])[["all", "train (train+val)", "test"]]
(props / props.sum() * 100).plot.bar(figsize=(10, 4), color=plt.cm.tab10.colors[:3])
plt.ylabel("% of the split"); plt.title("Class proportions", fontweight="bold")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

## 3. Channel statistics: EuroSAT vs. ImageNet
The student is normalized with the ImageNet statistics (report Sec. 5.1), as the teacher requires.
How far are EuroSAT's own statistics?

In [ ]:
cs = pd.DataFrame({"EuroSAT mean": stats["channel_stats"]["mean"],
                   "ImageNet mean": stats["imagenet"]["mean"],
                   "EuroSAT std": stats["channel_stats"]["std"],
                   "ImageNet std": stats["imagenet"]["std"]}, index=["R", "G", "B"])
display(cs)
# Scale of the student's input after ImageNet normalization: (x - m_in) / s_in
x_mean = (np.array(cs["EuroSAT mean"]) - np.array(cs["ImageNet mean"])) / np.array(cs["ImageNet std"])
x_std = np.array(cs["EuroSAT std"]) / np.array(cs["ImageNet std"])
display(pd.DataFrame({"normalized input mean": x_mean, "normalized input std": x_std},
                     index=["R", "G", "B"]).round(3))

## 4. Mean image and colour per class
How much does colour alone separate the classes?

In [ ]:
fig, axes = plt.subplots(1, len(classes), figsize=(len(classes) * 1.5, 2))
for k, name in enumerate(classes):
    axes[k].imshow(images[labels == k].mean(0).astype(np.uint8))
    axes[k].set_title(name, fontsize=8); axes[k].axis("off")
plt.suptitle("Mean image per class", fontweight="bold"); plt.tight_layout(); plt.show()

display(pd.DataFrame(stats["per_class_channel_mean"], index=["R", "G", "B"]).T.round(3))

In [ ]:
brightness = images.reshape(len(images), -1, 3).mean(1)          # per-image mean of R, G, B
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for c, (ax, ch) in enumerate(zip(axes, "RGB")):
    for k, name in enumerate(classes):
        ax.hist(brightness[labels == k, c], bins=50, histtype="step", density=True,
                color=plt.cm.tab10(k), label=name)
    ax.set_title(f"Per-image mean, channel {ch}", fontweight="bold"); ax.set_xlabel("0-255")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 5. Duplicates and degenerate images
Exact duplicates (identical pixels) across the train/test split would inflate test accuracy.

In [ ]:
print("exact duplicate groups:", len(stats["exact_duplicate_groups"]))
print("duplicate groups across train/test:", len(stats["cross_split_duplicate_groups"]))
print("degenerate images:", {k: len(v) for k, v in stats["degenerate"].items()},
      "| thresholds:", stats["thresholds"])
for g in stats["exact_duplicate_groups"][:5]:
    fig, axes = plt.subplots(1, len(g), figsize=(1.5 * len(g), 1.5))
    for ax, i in zip(np.atleast_1d(axes), g):
        ax.imshow(images[i]); ax.set_title(data["filenames"][i], fontsize=6); ax.axis("off")
    plt.show()

## 6. What the teacher sees
The ViT-S/16 teacher needs 224 px: each 64 px image is upsampled with bicubic interpolation
(report Sec. 3.2). Bicubic can overshoot, so values are clamped to [0, 1] here for display.

In [ ]:
idx = [rng.choice(np.flatnonzero(labels == k)) for k in range(0, len(classes), 3)]
x = torch.from_numpy(images[idx]).permute(0, 3, 1, 2).float() / 255
up = F.interpolate(x, size=config.TEACHER_RES, mode=config.TEACHER_UPSAMPLE, align_corners=False)
print(f"bicubic overshoot: min {up.min():.3f}, max {up.max():.3f}")
fig, axes = plt.subplots(2, len(idx), figsize=(3 * len(idx), 6))
for j, i in enumerate(idx):
    axes[0, j].imshow(images[i]); axes[0, j].set_title(f"{classes[labels[i]]}, 64 px", fontsize=9)
    axes[1, j].imshow(up[j].clamp(0, 1).permute(1, 2, 0)); axes[1, j].set_title("224 px bicubic", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

## Notes
Write here anything surprising, (the findings are in the report, Sec. 5.1).